# Exploratory Data Analysis - Olympic Countries Efficiency

This notebook performs comprehensive exploratory data analysis on the Olympic countries efficiency dataset.
It includes:
- Data loading and basic exploration
- Statistical summaries and distributions
- Missing value analysis
- Correlation analysis
- Visualizations
- Data quality assessment

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.data_loader import load_data, get_data_info
from src.visualization.plots import (
    plot_distributions, plot_correlation_heatmap, plot_income_group_analysis
)

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print('Libraries imported successfully!')

## 2. Load Data

In [ ]:
# Load the dataset
data_path = '../data/raw/olympic_countries_efficiency.csv'
df = load_data(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Overview

In [ ]:
# Get basic information
print("Dataset Info:")
print(f"Shape: {df.shape}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nColumn Names:")
print(df.columns.tolist())

## 4. Missing Values Analysis

In [ ]:
# Analyze missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

print("Missing Values Summary:")
print(missing_data)

# Visualize missing values
if len(missing_data) > 0:
    plt.figure(figsize=(10, 6))
    missing_data.sort_values('Missing_Percentage').plot(
        x='Column', y='Missing_Percentage', kind='barh', figsize=(10, 6)
    )
    plt.title('Missing Values Percentage by Column')
    plt.xlabel('Percentage (%)')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo missing values found!")

## 5. Statistical Summary

In [ ]:
# Statistical summary
print("Statistical Summary of Numerical Columns:")
df.describe().T

## 6. Categorical Variables Analysis

In [ ]:
# Analyze categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    print(f"\n{col} - Value Counts:")
    print(df[col].value_counts())
    print(f"Unique values: {df[col].nunique()}")

## 7. Income Group Analysis

In [ ]:
# Analyze by income group
if 'income_group' in df.columns:
    print("\nIncome Group Distribution:")
    print(df['income_group'].value_counts())
    
    print("\nMean Medals by Income Group:")
    print(df.groupby('income_group')['total_medals'].agg(['count', 'mean', 'std', 'min', 'max']))

## 8. Distribution Analysis

In [ ]:
# Plot distributions
plot_distributions(df, figsize=(15, 10))
plt.suptitle('Distribution of Key Variables', fontsize=16, y=1.00)
plt.show()

## 9. Correlation Analysis

In [ ]:
# Correlation analysis
plot_correlation_heatmap(df, figsize=(14, 10))
plt.title('Correlation Matrix - Olympic Efficiency Dataset', fontsize=14, pad=20)
plt.show()

## 10. Top Correlations with Target Variable

In [ ]:
# Top correlations with total_medals (target)
if 'total_medals' in df.columns:
    numeric_df = df.select_dtypes(include=[np.number])
    correlations = numeric_df.corr()['total_medals'].sort_values(ascending=False)
    
    print("\nTop 10 Features Correlated with Total Medals:")
    print(correlations.head(11))  # 11 to exclude the target itself
    
    # Visualize
    plt.figure(figsize=(10, 6))
    correlations.drop('total_medals').sort_values().plot(kind='barh', color='steelblue')
    plt.title('Feature Correlation with Total Medals')
    plt.xlabel('Correlation Coefficient')
    plt.tight_layout()
    plt.show()

## 11. Temporal Analysis

In [ ]:
# Temporal trends
if 'Year' in df.columns:
    print("\nOlympic Years in Dataset:")
    print(sorted(df['Year'].unique()))
    
    # Trends over time
    yearly_stats = df.groupby('Year').agg({
        'total_medals': ['sum', 'mean', 'std'],
        'NOC': 'count',
        'athletes_sent': 'mean'
    }).round(2)
    
    print("\nYearly Statistics:")
    print(yearly_stats)
    
    # Plot trend
    plt.figure(figsize=(12, 6))
    yearly_medals = df.groupby('Year')['total_medals'].sum()
    yearly_medals.plot(marker='o', linewidth=2, markersize=8)
    plt.title('Total Medals by Olympic Year')
    plt.xlabel('Year')
    plt.ylabel('Total Medals')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 12. Key Insights and Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS FROM EDA")
print("="*60)

print(f"\n1. Dataset Size: {df.shape[0]} records across {df.shape[1]} features")

print(f"\n2. Time Period: {df['Year'].min()} to {df['Year'].max()}")

print(f"\n3. Countries: {df['NOC'].nunique()} countries represented")

if 'total_medals' in df.columns:
    print(f"\n4. Medal Statistics:")
    print(f"   - Mean: {df['total_medals'].mean():.2f}")
    print(f"   - Median: {df['total_medals'].median():.2f}")
    print(f"   - Max: {df['total_medals'].max():.2f}")
    print(f"   - Countries with no medals: {(df['total_medals'] == 0).sum()}")

if 'income_group' in df.columns:
    print(f"\n5. Income Distribution:")
    for income in df['income_group'].unique():
        count = (df['income_group'] == income).sum()
        avg_medals = df[df['income_group'] == income]['total_medals'].mean()
        print(f"   - {income}: {count} records, avg {avg_medals:.2f} medals")

if 'gdp_per_capita' in df.columns and 'total_medals' in df.columns:
    corr = df['gdp_per_capita'].corr(df['total_medals'])
    print(f"\n6. GDP-Medals Correlation: {corr:.3f}")

print(f"\n7. Data Quality: {100 - (df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100):.1f}% complete")

print("\n" + "="*60)